# Cold Email Personalization Engine

| Field | Details |
|-------|--------|
| **Category** | Job Search |
| **Difficulty** | Advanced |
| **Primary Tools** | n8n, Claude MCP, Gmail MCP, Airtable MCP, Hunter.io API |

## 📋 Project Description

Input a list of hiring managers → scrape their LinkedIn/company page → Claude MCP writes a hyper-personalized cold email → sends via Gmail → tracks opens.

## 💼 Why This Gets You Hired

Combines lead enrichment, AI writing, and email automation — a trifecta of modern sales/outreach skills.

---


In [ ]:
import smtplib
import pandas as pd
import time
import re
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders


In [ ]:

EMAIL = "dronabopche489@gmail.com"
PASSWORD = "qducrplxnhizjgfq"


In [ ]:

START_FROM_ROW = 1800   # 🔹 change to resume (e.g., 50)


In [ ]:
df = pd.read_csv("mail.csv")

In [ ]:

# ✅ email validator
def is_valid_email(email):
    pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return re.match(pattern, str(email)) is not None

# ✅ dynamic email body based on title
def create_body(name, company, title):
    title_lower = title.lower()

    if "hr" in title_lower or "recruit" in title_lower:
        line = f"I would love to connect with you regarding opportunities at {company}."
    elif "engineer" in title_lower or "developer" in title_lower:
        line = f"I’m particularly interested in contributing to technical teams at {company}."
    else:
        line = f"I would love to contribute to {company}."

    return f"""
Hi {name},

I am a 3rd-year student BTech From North Eastern Hill University, specializing in AI/ML and looking for internship opportunities.

{line}

Resume attached.

Thanks
"""


In [ ]:

def send_email(to_email, name, company, title):
    msg = MIMEMultipart()
    msg['From'] = EMAIL
    msg['To'] = to_email
    msg['Subject'] = "Application for AI/ML Internship"

    body = create_body(name, company, title)
    msg.attach(MIMEText(body, 'plain'))

    # attach resume
    filename = "resume.pdf"
    with open(filename, "rb") as f:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(f.read())
        encoders.encode_base64(part)
        part.add_header('Content-Disposition', f"attachment; filename={filename}")
        msg.attach(part)

    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
        server.login(EMAIL, PASSWORD)
        server.send_message(msg)

# 🚀 main loop
count = 0

for idx, row in df.iterrows():
    if idx < START_FROM_ROW:
        continue  # ⏩ skip processed rows

    # ✅ clean data
    email = str(row['Email']).strip()
    name = str(row['Name']).strip()
    company = str(row['Company']).strip()
    title = str(row['Title']).strip()

    # ❌ missing data check
    if pd.isna(row['Email']) or pd.isna(row['Name']):
        print(f"Skipping row {idx} due to missing data")
        continue

    # ❌ invalid email format
    if not is_valid_email(email):
        print(f"Invalid email skipped: {email}")
        continue

    try:
        send_email(email, name, company, title)
        print(f"Sent to {email} (Row {idx})")

        count += 1
        time.sleep(360)  # 6 min wait

        #cooldown 10 emails
        if count % 10 == 0:
            print("Cooldown...")
            time.sleep(1800)  #30 minbreak

    except Exception as e:
        print(f"Failed for {email} (Row {idx}): {e}")

        with open("failed_emails.txt", "a") as f:
            f.write(f"{email} | Row: {idx} | Error: {e}\n")

        continue